In [35]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from IPython.display import display
import gender_guesser.detector as gender
from ast import literal_eval

import imageio.v2 as imageio
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MaxNLocator
import matplotlib.colors as colors

plt.rcParams['figure.figsize'] = (16, 9)
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.family'] = 'Helvetica'

# ======================================================================
# 1. Load data from Excel
# ======================================================================
df = pd.read_excel('../data/dimensions/api/raw/combined/202511/df_dimensions.xlsx', index_col=0)

# Ensure 'year' exists and is numeric
if 'year' not in df.columns:
    raise ValueError("'year' column not found in the dataset.")
df['year'] = pd.to_numeric(df['year'], errors='coerce')

# ======================================================================
# 2. Helper
# ======================================================================

def split_list_field(value):
    if pd.isna(value):
        return []
    s = str(value)

    # Try to parse as Python literal list
    try:
        parsed = literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass

    # Fallback: split on common delimiters
    for sep in [';', '|']:
        if sep in s:
            return [x.strip() for x in s.split(sep) if x.strip()]

    # Last resort: single value
    s = s.strip()
    return [s] if s else []

# ======================================================================
# 3. Countries / institutions tables
# ======================================================================

# --- Countries exploded table ---
countries_df = (
    df[['id', 'year', 'research_org_country_names']]
    .assign(country_list=lambda d: d['research_org_country_names'].apply(split_list_field))
    .explode('country_list')
    .rename(columns={'country_list': 'country'})
)

countries_df = countries_df.dropna(subset=['country', 'year'])
countries_df['year'] = countries_df['year'].astype(int)

# Normalise country names to match Natural Earth "ADMIN" field
name_map = {
    # United States
    'USA': 'United States of America',
    'US': 'United States of America',
    'U.S.': 'United States of America',
    'U.S.A.': 'United States of America',
    'United States': 'United States of America',
    'United States of America': 'United States of America',

    # United Kingdom
    'UK': 'United Kingdom',
    'U.K.': 'United Kingdom',
    'Great Britain': 'United Kingdom',
    'England': 'United Kingdom',
    'United Kingdom': 'United Kingdom',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',

    # Example mapping; adjust if needed for your shapefile
    'Russian Federation': 'Russia',
}

countries_df['country'] = countries_df['country'].astype(str).str.strip()
countries_df['country_norm'] = countries_df['country'].replace(name_map)
countries_df['country_norm'] = countries_df['country_norm'].fillna(countries_df['country'])

# Use normalised names going forward
countries_df['country'] = countries_df['country_norm']
countries_df = countries_df.drop(columns=['country_norm'])

print("\nExploded (paper, year, country) table:")
display(countries_df.head())

# Overall country counts
country_counts = (
    countries_df
    .groupby('country')
    .agg(n_pubs=('id', 'nunique'))
    .reset_index()
    .sort_values('n_pubs', ascending=False)
)

print("\nTop countries by number of UKBB publications:")
display(country_counts.head(10))

# Institutions table
if 'research_org_names' in df.columns:
    orgs_df = (
        df[['id', 'year', 'research_org_names']]
        .assign(org_list=lambda d: d['research_org_names'].apply(split_list_field))
        .explode('org_list')
        .rename(columns={'org_list': 'org_name'})
    )

    orgs_df = orgs_df.dropna(subset=['org_name', 'year'])
    orgs_df['year'] = orgs_df['year'].astype(int)

    org_counts = (
        orgs_df.groupby('org_name')
        .agg(n_pubs=('id', 'nunique'))
        .reset_index()
        .sort_values('n_pubs', ascending=False)
    )

    print("\nTop institutions by number of UKBB publications:")
    display(org_counts.head(10))
else:
    print("\n'research_org_names' column not found — cannot print institution table.")
    orgs_df = None

# ======================================================================
# 4. Choropleth base + uptake curves
# ======================================================================

# Natural Earth 1:110m countries shapefile
world_url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(world_url)

print("\nWorld shapefile example:")
display(world[['ADMIN', 'ISO_A3_EH']].head())

# Overall counts merged to world (for static map only)
world_counts = world.merge(
    country_counts,
    how='left',
    left_on='ADMIN',
    right_on='country'
)
world_counts['n_pubs'] = world_counts['n_pubs'].fillna(0)

print("\nTop countries by number of UKBB publications (recomputed):")
display(country_counts.head(10))

# Cumulative distinct countries over time
year_country = (
    countries_df[['year', 'country']]
    .drop_duplicates()
    .sort_values(['year', 'country'])
)

years = sorted(year_country['year'].unique())
cum_counts = []
seen = set()

for y in years:
    subset = year_country[year_country['year'] == y]['country']
    seen.update(subset)
    cum_counts.append({'year': y, 'cum_countries': len(seen)})

uptake_df = pd.DataFrame(cum_counts)

# Cumulative institutions
if orgs_df is not None:
    year_org = (
        orgs_df[['year', 'org_name']]
        .drop_duplicates()
        .sort_values(['year', 'org_name'])
    )

    years_org = sorted(year_org['year'].unique())
    cum_counts_org = []
    seen_orgs = set()

    for y in years_org:
        subset = year_org[year_org['year'] == y]['org_name']
        seen_orgs.update(subset)
        cum_counts_org.append({'year': y, 'cum_orgs': len(seen_orgs)})

    uptake_org_df = pd.DataFrame(cum_counts_org)
else:
    print("\n'research_org_names' column not found - skipping institution uptake analysis.")
    uptake_org_df = pd.DataFrame(columns=['year', 'cum_orgs'])

# ---- Global y-limits for line plots (countries / institutions) ----
ymax_countries = uptake_df['cum_countries'].max() if not uptake_df.empty else 0
ymax_orgs = uptake_org_df['cum_orgs'].max() if not uptake_org_df.empty else 0
if ymax_countries <= 0:
    ymax_countries = 1
if ymax_orgs <= 0:
    ymax_orgs = 1

# ======================================================================
# 5. Static figure (overall geography, non-cumulative)
# ======================================================================

plt.close('all')

ZERO_COLOR = "#E5E5E5"

world_counts_no_ant = world_counts[world_counts["ADMIN"] != "Antarctica"].copy()

vals_static = world_counts_no_ant['n_pubs'].to_numpy(dtype=float)
pos_static = vals_static[vals_static > 0]
if pos_static.size == 0:
    raise ValueError("No non-zero country publication counts found in world_counts.")

cmap_static = plt.colormaps.get_cmap("cividis")
norm_static = colors.Normalize(vmin=pos_static.min(), vmax=pos_static.max())

# Split into zero / non-zero layers
world_static_zero = world_counts_no_ant[world_counts_no_ant['n_pubs'] <= 0]
world_static_nonzero = world_counts_no_ant[world_counts_no_ant['n_pubs'] > 0]

fig = plt.figure(figsize=(16, 7.5), dpi=300)
gs = GridSpec(
    2, 4,
    figure=fig,
    width_ratios=[0.15, 4.5, 1.6, 0.1],
    height_ratios=[1, 1],
    wspace=0.05,
    hspace=0.3
)

# Map panel
ax_map = fig.add_subplot(gs[:, 1])

# Zero-valued countries in grey
world_static_zero.plot(
    ax=ax_map,
    color=ZERO_COLOR,
    edgecolor="black",
    linewidth=0.3
)

# Non-zero countries with colormap
world_static_nonzero.plot(
    ax=ax_map,
    column='n_pubs',
    cmap=cmap_static,
    vmin=pos_static.min(),
    vmax=pos_static.max(),
    edgecolor="black",
    linewidth=0.3
)

ax_map.set_aspect("auto")
ax_map.margins(0)
ax_map.set_anchor('W')
ax_map.set_axis_off()

# Colourbar
ax_cbar = fig.add_subplot(gs[:, 0])
sm = plt.cm.ScalarMappable(
    cmap=cmap_static,
    norm=norm_static
)
sm._A = []
cbar = fig.colorbar(sm, cax=ax_cbar, orientation='vertical')
cbar.ax.yaxis.set_ticks_position('left')
cbar.ax.yaxis.set_label_position('left')
cbar.set_label("International Distribution of UK Biobank Using Authors", fontsize=12)

# Top right — cumulative countries
ax1 = fig.add_subplot(gs[0, 2])

# Fill under curve
ax1.fill_between(
    uptake_df['year'], uptake_df['cum_countries'],
    color='#D4AF37', alpha=0.3
)

ax1.plot(
    uptake_df['year'], uptake_df['cum_countries'],
    marker='o', linewidth=2, markersize=6,
    markeredgecolor='k', markerfacecolor='#D4AF37', color='#345995'
)
ax1.yaxis.tick_right()
ax1.yaxis.set_label_position("right")
ax1.set_ylabel("Cumulative Countries Observed")
ax1.spines['left'].set_visible(False)
ax1.spines['top'].set_visible(False)
ax1.tick_params(axis='y', right=True, left=False)
ax1.tick_params(axis='x', bottom=True, top=False)
ax1.xaxis.set_major_locator(MaxNLocator(integer=True, prune='both', nbins=5))
ax1.set_ylim(0, ymax_countries+3)  # fixed global y-limit

# Bottom right — cumulative institutions
ax2 = fig.add_subplot(gs[1, 2])

# Fill under curve
ax2.fill_between(
    uptake_org_df['year'], uptake_org_df['cum_orgs'],
    color='#D4AF37', alpha=0.3
)

ax2.plot(
    uptake_org_df['year'], uptake_org_df['cum_orgs'],
    marker='o', linewidth=2, markersize=6,
    markeredgecolor='k', markerfacecolor='#D4AF37', color='#345995'
)
ax2.yaxis.tick_right()
ax2.yaxis.set_label_position("right")
ax2.set_ylabel("Cumulative Institutions Observed")
ax2.set_xlabel("Year")
ax2.spines['left'].set_visible(False)
ax2.spines['top'].set_visible(False)
ax2.tick_params(axis='y', right=True, left=False)
ax2.tick_params(axis='x', bottom=True, top=False)
ax2.xaxis.set_major_locator(MaxNLocator(integer=True, prune='both', nbins=5))
ax2.set_ylim(0, ymax_orgs+200)  # fixed global y-limit

# Spacer
ax_blank = fig.add_subplot(gs[:, 3])
ax_blank.set_axis_off()

# Save static figure
out_dir = "../output/figures"
os.makedirs(out_dir, exist_ok=True)

plt.savefig(os.path.join(out_dir, "geography.pdf"), bbox_inches='tight')
plt.savefig(os.path.join(out_dir, "geography.svg"), bbox_inches='tight')
plt.savefig(os.path.join(out_dir, "geography.png"), bbox_inches='tight', dpi=300)

plt.close(fig)

# ======================================================================
# 6. Animated GIF over years (cumulative per country) + first/last-year SVGs
# ======================================================================

# 6.1 Cumulative country counts per year (full country–year grid)
per_year = (
    countries_df
    .groupby(['country', 'year'])
    .agg(n_pubs=('id', 'nunique'))
    .reset_index()
)

all_years = np.sort(countries_df['year'].unique())
all_countries = np.sort(countries_df['country'].unique())
full_index = pd.MultiIndex.from_product(
    [all_countries, all_years],
    names=['country', 'year']
)

per_year_full = (
    per_year
    .set_index(['country', 'year'])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

per_year_full = per_year_full.sort_values(['country', 'year'])
per_year_full['cum_n'] = (
    per_year_full
    .groupby('country')['n_pubs']
    .cumsum()
)

year_country_cum = per_year_full[['year', 'country', 'cum_n']].reset_index(drop=True)

vals_anim_all = year_country_cum['cum_n'].to_numpy(dtype=float)
pos_anim = vals_anim_all[vals_anim_all > 0]
if pos_anim.size == 0:
    raise ValueError("No non-zero country publication counts found. Cannot build animated map.")

vmin_anim, vmax_anim = pos_anim.min(), pos_anim.max()
cmap_anim = plt.colormaps.get_cmap("cividis")
norm_anim = colors.Normalize(vmin=vmin_anim, vmax=vmax_anim)

# Base geometry for animation: shapes only, no data columns
world_base = world[world["ADMIN"] != "Antarctica"][['ADMIN', 'geometry']].copy()

years_all = np.sort(year_country_cum['year'].unique())

def make_geography_figure_for_year(y: int):
    """
    Build the full 2x4 grid figure for a given year y:
      - Map: cumulative counts per country at year y
      - Upper right line: cumulative countries up to year y
      - Lower right line: cumulative institutions up to year y
    """
    # Map data for year y (cumulative counts)
    cc_y = year_country_cum[year_country_cum['year'] == y]

    world_y = world_base.merge(
        cc_y[['country', 'cum_n']],
        how='left',
        left_on='ADMIN',
        right_on='country'
    )

    world_y['cum_n'] = world_y['cum_n'].fillna(0)

    # Split into zero / non-zero layers for colouring
    world_anim_zero = world_y[world_y['cum_n'] <= 0]
    world_anim_nonzero = world_y[world_y['cum_n'] > 0]

    uptake_y = uptake_df[uptake_df['year'] <= y]
    uptake_org_y = uptake_org_df[uptake_org_df['year'] <= y]

    fig = plt.figure(figsize=(16, 7.5), dpi=300)
    gs = GridSpec(
        2, 4,
        figure=fig,
        width_ratios=[0.15, 4.5, 1.6, 0.1],
        height_ratios=[1, 1],
        wspace=0.05,
        hspace=0.3
    )

    # Map panel
    ax_map = fig.add_subplot(gs[:, 1])

    # Zero-valued countries in grey
    if not world_anim_zero.empty:
        world_anim_zero.plot(
            ax=ax_map,
            color=ZERO_COLOR,
            edgecolor="black",
            linewidth=0.3
        )

    # Non-zero cumulative counts with global colour scale
    if not world_anim_nonzero.empty:
        world_anim_nonzero.plot(
            ax=ax_map,
            column='cum_n',
            cmap=cmap_anim,
            vmin=vmin_anim,
            vmax=vmax_anim,
            edgecolor="black",
            linewidth=0.3
        )

    ax_map.set_aspect("auto")
    ax_map.margins(0)
    ax_map.set_anchor('W')
    ax_map.set_axis_off()

    # Year annotation, bottom-left of map (just to right of colourbar)
    ax_map.text(
        0.03, 0.04,
        f"Year: {y}",
        transform=ax_map.transAxes,
        ha="left",
        va="bottom",
        fontsize=12,
        fontweight="bold",
        bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", pad=3),
    )

    # Colourbar with global norm
    ax_cbar = fig.add_subplot(gs[:, 0])
    sm = plt.cm.ScalarMappable(cmap=cmap_anim, norm=norm_anim)
    sm._A = []
    cbar = fig.colorbar(sm, cax=ax_cbar, orientation='vertical')
    cbar.ax.yaxis.set_ticks_position('left')
    cbar.ax.yaxis.set_label_position('left')
    cbar.set_label("International Distribution of UK Biobank Using Authors", fontsize=12)

    # Top right: cumulative countries up to year y
    ax1 = fig.add_subplot(gs[0, 2])

    # Fill under curve (truncated to year y)
    ax1.fill_between(
        uptake_y['year'], uptake_y['cum_countries'],
        color='#D4AF37', alpha=0.3
    )

    ax1.plot(
        uptake_y['year'], uptake_y['cum_countries'],
        marker='o', linewidth=2, markersize=6,
        markeredgecolor='k', markerfacecolor='#D4AF37', color='#345995'
    )
    ax1.yaxis.tick_right()
    ax1.yaxis.set_label_position("right")
    ax1.set_ylabel("Cumulative Countries Observed")
    ax1.spines['left'].set_visible(False)
    ax1.spines['top'].set_visible(False)
    ax1.tick_params(axis='y', right=True, left=False)
    ax1.tick_params(axis='x', bottom=True, top=False)
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True, prune='both', nbins=5))
    ax1.set_ylim(0, ymax_countries+3)  # fixed global y-limit

    # Bottom right: cumulative institutions up to year y
    ax2 = fig.add_subplot(gs[1, 2])

    # Fill under curve (truncated to year y)
    ax2.fill_between(
        uptake_org_y['year'], uptake_org_y['cum_orgs'],
        color='#D4AF37', alpha=0.3
    )

    ax2.plot(
        uptake_org_y['year'], uptake_org_y['cum_orgs'],
        marker='o', linewidth=2, markersize=6,
        markeredgecolor='k', markerfacecolor='#D4AF37', color='#345995'
    )
    ax2.yaxis.tick_right()
    ax2.yaxis.set_label_position("right")
    ax2.set_ylabel("Cumulative Institutions Observed")
    ax2.set_xlabel("Year")
    ax2.spines['left'].set_visible(False)
    ax2.spines['top'].set_visible(False)
    ax2.tick_params(axis='y', right=True, left=False)
    ax2.tick_params(axis='x', bottom=True, top=False)
    ax2.xaxis.set_major_locator(MaxNLocator(integer=True, prune='both', nbins=5))
    ax2.set_ylim(0, ymax_orgs+200)  # fixed global y-limit

    # Spacer
    ax_blank = fig.add_subplot(gs[:, 3])
    ax_blank.set_axis_off()

    return fig

frames = []
first_year = int(years_all[0])
last_year = int(years_all[-1])

for y in years_all:
    fig = make_geography_figure_for_year(int(y))

    # Save SVG for first and last year (full figure)
    if y == first_year:
        fig.savefig(
            os.path.join(out_dir, f"geography_{first_year}.svg"),
            bbox_inches='tight'
        )
    if y == last_year:
        fig.savefig(
            os.path.join(out_dir, f"geography_{last_year}.svg"),
            bbox_inches='tight'
        )

    # Render figure to RGB array for GIF
    canvas = FigureCanvas(fig)
    canvas.draw()
    width, height = fig.canvas.get_width_height()
    frame = np.frombuffer(canvas.tostring_rgb(), dtype='uint8')
    frame = frame.reshape((height, width, 3))

    frames.append(frame)
    plt.close(fig)

# Save animated GIF with 2-second interval per frame
gif_path = os.path.join(out_dir, "geography_evolving.gif")
imageio.mimsave(gif_path, frames, duration=1000.0)

print(f"Animated GIF written to: {gif_path}")
print(f"First-year SVG: {os.path.join(out_dir, f'geography_{first_year}.svg')}")
print(f"Last-year SVG: {os.path.join(out_dir, f'geography_{last_year}.svg')}")



Exploded (paper, year, country) table:


,id,year,research_org_country_names,country
0,pub.1142697354,2021,"['China', 'Australia']",China
0,pub.1142697354,2021,"['China', 'Australia']",Australia
1,pub.1107515670,2018,"['United Kingdom', 'Norway']",United Kingdom
1,pub.1107515670,2018,"['United Kingdom', 'Norway']",Norway
2,pub.1111460082,2019,"['United States', 'United Kingdom']",United States of America



Top countries by number of UKBB publications:


,country,n_pubs
117,United States of America,4228
116,United Kingdom,4205
21,China,3666
3,Australia,1269
106,Sweden,955
39,Germany,944
75,Netherlands,942
19,Canada,722
28,Denmark,520
36,France,487



Top institutions by number of UKBB publications:


...


World shapefile example:


,ADMIN,ISO_A3_EH
0,Fiji,FJI
1,United Republic of Tanzania,TZA
2,Western Sahara,ESH
3,Canada,CAN
4,United States of America,USA



Top countries by number of UKBB publications (recomputed):


,country,n_pubs
117,United States of America,4228
116,United Kingdom,4205
21,China,3666
3,Australia,1269
106,Sweden,955
39,Germany,944
75,Netherlands,942
19,Canada,722
28,Denmark,520
36,France,487


Animated GIF written to: ../output/figures/geography_evolving.gif
First-year SVG: ../output/figures/geography_2013.svg
Last-year SVG: ../output/figures/geography_2025.svg
